In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
df = pd.read_excel('df_all_features.xlsx')

## Гипотеза 1. Соответствие статусности локации и стоимости абонемента

Если цена абонемента слишком высока для доходов населения вокруг, то клуб закрывается

Прокси:
- 'rent_2026_price_per_m2_mean_15ped'
- 'buy_2026_price_per_m2_mean_15ped',
- 'buy_2021_price_per_m2_mean_15ped',
- 'incomes_2025_avg_hcs_cost_pop_mean_15ped'
- 'buy_2021_2026_abs_15ped',
- 'buy_2021_2026_pct_15ped'
- 'cafe_checks_avg_price_mean_15ped',
- 'supermarkets4rich_count_15ped'

- 'subs_month'
- 'subs_year'

In [23]:
df1m = df.dropna(subset=['subs_month'])
df1y = df.dropna(subset=['subs_year'])
print(df1m.shape)
print(df1y.shape)

(2325, 333)
(1196, 333)


In [20]:
df.dropna(subset=['subs_month']).shape

(2325, 333)

In [24]:
df1m = df1m[['last_seen',
           'rent_2026_price_per_m2_mean_15ped',
     'buy_2026_price_per_m2_mean_15ped',
     'buy_2021_price_per_m2_mean_15ped',
     'incomes_2025_avg_hcs_cost_pop_mean_15ped',
     'buy_2021_2026_abs_15ped',
     'buy_2021_2026_pct_15ped',
     'cafe_checks_avg_price_mean_15ped',
     'supermarkets4rich_count_15ped',
    'subs_month',
     'subs_year',
    'is_closed']]
df1m['is_closed'].value_counts()

is_closed
0    1818
1     507
Name: count, dtype: int64

In [25]:
df1y = df1y[['last_seen',
           'rent_2026_price_per_m2_mean_15ped',
     'buy_2026_price_per_m2_mean_15ped',
     'buy_2021_price_per_m2_mean_15ped',
     'incomes_2025_avg_hcs_cost_pop_mean_15ped',
     'buy_2021_2026_abs_15ped',
     'buy_2021_2026_pct_15ped',
     'cafe_checks_avg_price_mean_15ped',
     'supermarkets4rich_count_15ped',
    'subs_month',
     'subs_year',
    'is_closed']]
df1y['is_closed'].value_counts()

is_closed
0    972
1    224
Name: count, dtype: int64

In [26]:
round(df1m.groupby(by='is_closed')[['subs_month']].mean()/1000, 2)

,subs_month
is_closed,
0,7.81
1,5.62


In [27]:
round(df1y.groupby(by='is_closed')[['subs_year']].mean()/1000, 2)

,subs_year
is_closed,
0,64.47
1,40.96


In [28]:
round(df1m.groupby(by='last_seen')[['subs_month']].mean()/1000, 2)

,subs_month
last_seen,
2024-02-22,4.65
2024-07-25,5.24
2024-12-13,5.26
2025-02-07,5.05
2025-08-27,8.19
2025-11-17,6.71
2026-02-17,4.73
2026-03-24,7.81


In [29]:
round(df1y.groupby(by='last_seen')[['subs_year']].mean()/1000, 2)

,subs_year
last_seen,
2024-02-22,21.77
2024-07-25,37.41
2024-12-13,47.01
2025-02-07,32.45
2025-08-27,43.47
2025-11-17,60.65
2026-02-17,43.64
2026-03-24,64.47


In [30]:
df1m['last_seen'].value_counts()

last_seen
2026-03-24    1818
2025-02-07     126
2024-07-25     117
2025-11-17      69
2025-08-27      58
2026-02-17      56
2024-02-22      44
2024-12-13      37
Name: count, dtype: int64

### Рассмотрм месячный абонемент

In [31]:
# ---------------------------------------------------------
# 0. КОРРЕКТИРОВКА НА ИНФЛЯЦИЮ (Temporal Bias Adjustment)
# ---------------------------------------------------------
# Переводим дату в формат datetime, чтобы с ней можно было считать разницу
df1 = df1m.copy()
df1['last_seen'] = pd.to_datetime(df1['last_seen'])

# Назначаем "базовую дату" - последнюю дату в датасете (скорее всего март 2026)
base_date = df1['last_seen'].max()

# Задаем среднюю годовую инфляцию в фитнес-индустрии (например, 15% = 0.15)
# Эту цифру можно обосновать в дипломе ссылкой на индекс цен на услуги Росстата
annual_inflation = 0.14 

# Считаем, сколько лет (или долей года) прошло с момента last_seen до base_date
df1['years_diff'] = (base_date - df1['last_seen']).dt.days / 365.25

# Приводим цены абонементов к уровню 2026 года по формуле сложных процентов
df1['subs_month_adj'] = df1['subs_month'] * ((1 + annual_inflation) ** df1['years_diff'])
if 'subs_year' in df1.columns:
    df1['subs_year_adj'] = df1['subs_year'] * ((1 + annual_inflation) ** df1['years_diff'])

# ---------------------------------------------------------
# 1. ФИЧИ-ИНЖИНИРИНГ: Создаем показатели "дороговизны" (уже с учетом инфляции)
# ---------------------------------------------------------
wealth_proxies = [
    'incomes_2025_avg_hcs_cost_pop_mean_15ped',
    'rent_2026_price_per_m2_mean_15ped',
    'buy_2026_price_per_m2_mean_15ped',
    'cafe_checks_avg_price_mean_15ped'
]

ratio_cols = []

# Считаем отношение ИНФЛИРОВАННОЙ цены абонемента к прокси богатства
for col in wealth_proxies:
    ratio_col_name = f'ratio_subs_month_adj_to_{col}' # Обратите внимание: используем adj
    df1[ratio_col_name] = np.where(
        df1[col] > 0, 
        df1['subs_month_adj'] / df1[col], 
        np.nan
    )
    ratio_cols.append(ratio_col_name)

# Очищаем датафрейм от пустых значений для корректной работы тестов
df_clean = df1.replace([np.inf, -np.inf], np.nan).dropna(subset=ratio_cols + ['is_closed'])

# ---------------------------------------------------------
# 2. СТАТИСТИЧЕСКОЕ ТЕСТИРОВАНИЕ (U-критерий Манна-Уитни)
# ---------------------------------------------------------
print("=== Результаты статистического теста Манна-Уитни (С ПОПРАВКОЙ НА ИНФЛЯЦИЮ) ===\n")
print("H0: Распределения равны.")
print("H1: Абонементы закрытых объектов 'относительно' ДЕШЕВЛЕ (с учетом приведения цен к 2026 г.).\n")

for col in ratio_cols:
    closed = df_clean[df_clean['is_closed'] == 1][col]
    open_obj = df_clean[df_clean['is_closed'] == 0][col]
    
    # Проверяем, действительно ли closed статистически ЗНАЧИМО меньше, чем open
    stat, p_value = mannwhitneyu(closed, open_obj, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее у открытых (0): {open_obj.mean():.4f}")
    print(f"Среднее у закрытых (1): {closed.mean():.4f}")
    print(f"P-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые объекты были значимо дешевле (даже после выравнивания инфляции).")
    else:
        print("❌ Значимых различий нет (вся разница объяснялась инфляцией).")
    print("-" * 60)

# ---------------------------------------------------------
# 3. ДОПОЛНИТЕЛЬНО: Матрица корреляций Спирмена
# ---------------------------------------------------------
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
# Смотрим корреляцию уже для скорректированных цен
cols_to_corr = ['subs_month_adj', 'subs_year_adj', 'supermarkets4rich_count_15ped'] + ratio_cols

existing_cols = [c for c in cols_to_corr if c in df_clean.columns]
correlations = df_clean[existing_cols + ['is_closed']].corr(method='spearman')['is_closed'].sort_values()

print(correlations.drop('is_closed'))

=== Результаты статистического теста Манна-Уитни (С ПОПРАВКОЙ НА ИНФЛЯЦИЮ) ===

H0: Распределения равны.
H1: Абонементы закрытых объектов 'относительно' ДЕШЕВЛЕ (с учетом приведения цен к 2026 г.).

Метрика: ratio_subs_month_adj_to_incomes_2025_avg_hcs_cost_pop_mean_15ped
Среднее у открытых (0): 0.0037
Среднее у закрытых (1): 0.0019
P-value: 0.032766
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые объекты были значимо дешевле (даже после выравнивания инфляции).
------------------------------------------------------------
Метрика: ratio_subs_month_adj_to_rent_2026_price_per_m2_mean_15ped
Среднее у открытых (0): 4.1775
Среднее у закрытых (1): 3.3960
P-value: 0.056759
❌ Значимых различий нет (вся разница объяснялась инфляцией).
------------------------------------------------------------
Метрика: ratio_subs_month_adj_to_buy_2026_price_per_m2_mean_15ped
Среднее у открытых (0): 0.0170
Среднее у закрытых (1): 0.0139
P-value: 0.086954
❌ Значимых различий нет (вся разница объяснялась инфляцией).
---------

### Рассмотрим годовой абонемент

In [32]:
# ---------------------------------------------------------
# 0. КОРРЕКТИРОВКА НА ИНФЛЯЦИЮ (Temporal Bias Adjustment)
# ---------------------------------------------------------
# Переводим дату в формат datetime, чтобы с ней можно было считать разницу
df1 = df1y.copy()
df1['last_seen'] = pd.to_datetime(df1['last_seen'])

# Назначаем "базовую дату" - последнюю дату в датасете (скорее всего март 2026)
base_date = df1['last_seen'].max()

# Задаем среднюю годовую инфляцию в фитнес-индустрии (например, 15% = 0.15)
# Эту цифру можно обосновать в дипломе ссылкой на индекс цен на услуги Росстата
annual_inflation = 0.14 

# Считаем, сколько лет (или долей года) прошло с момента last_seen до base_date
df1['years_diff'] = (base_date - df1['last_seen']).dt.days / 365.25

# Приводим цены абонементов к уровню 2026 года по формуле сложных процентов
df1['subs_month_adj'] = df1['subs_month'] * ((1 + annual_inflation) ** df1['years_diff'])
if 'subs_year' in df1.columns:
    df1['subs_year_adj'] = df1['subs_year'] * ((1 + annual_inflation) ** df1['years_diff'])

# ---------------------------------------------------------
# 1. ФИЧИ-ИНЖИНИРИНГ: Создаем показатели "дороговизны" (уже с учетом инфляции)
# ---------------------------------------------------------
wealth_proxies = [
    'incomes_2025_avg_hcs_cost_pop_mean_15ped',
    'rent_2026_price_per_m2_mean_15ped',
    'buy_2026_price_per_m2_mean_15ped',
    'cafe_checks_avg_price_mean_15ped'
]

ratio_cols = []

# Считаем отношение ИНФЛИРОВАННОЙ цены абонемента к прокси богатства
for col in wealth_proxies:
    ratio_col_name = f'ratio_subs_year_adj_to_{col}' # Обратите внимание: используем adj
    df1[ratio_col_name] = np.where(
        df1[col] > 0, 
        df1['subs_year_adj'] / df1[col], 
        np.nan
    )
    ratio_cols.append(ratio_col_name)

# Очищаем датафрейм от пустых значений для корректной работы тестов
df_clean = df1.replace([np.inf, -np.inf], np.nan).dropna(subset=ratio_cols + ['is_closed'])

# ---------------------------------------------------------
# 2. СТАТИСТИЧЕСКОЕ ТЕСТИРОВАНИЕ (U-критерий Манна-Уитни)
# ---------------------------------------------------------
print("=== Результаты статистического теста Манна-Уитни (С ПОПРАВКОЙ НА ИНФЛЯЦИЮ) ===\n")
print("H0: Распределения равны.")
print("H1: Абонементы закрытых объектов 'относительно' ДЕШЕВЛЕ (с учетом приведения цен к 2026 г.).\n")

for col in ratio_cols:
    closed = df_clean[df_clean['is_closed'] == 1][col]
    open_obj = df_clean[df_clean['is_closed'] == 0][col]
    
    # Проверяем, действительно ли closed статистически ЗНАЧИМО меньше, чем open
    stat, p_value = mannwhitneyu(closed, open_obj, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее у открытых (0): {open_obj.mean():.4f}")
    print(f"Среднее у закрытых (1): {closed.mean():.4f}")
    print(f"P-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые объекты были значимо дешевле (даже после выравнивания инфляции).")
    else:
        print("❌ Значимых различий нет (вся разница объяснялась инфляцией).")
    print("-" * 60)

# ---------------------------------------------------------
# 3. ДОПОЛНИТЕЛЬНО: Матрица корреляций Спирмена
# ---------------------------------------------------------
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
# Смотрим корреляцию уже для скорректированных цен
cols_to_corr = ['subs_month_adj', 'subs_year_adj', 'supermarkets4rich_count_15ped'] + ratio_cols

existing_cols = [c for c in cols_to_corr if c in df_clean.columns]
correlations = df_clean[existing_cols + ['is_closed']].corr(method='spearman')['is_closed'].sort_values()

print(correlations.drop('is_closed'))

=== Результаты статистического теста Манна-Уитни (С ПОПРАВКОЙ НА ИНФЛЯЦИЮ) ===

H0: Распределения равны.
H1: Абонементы закрытых объектов 'относительно' ДЕШЕВЛЕ (с учетом приведения цен к 2026 г.).

Метрика: ratio_subs_year_adj_to_incomes_2025_avg_hcs_cost_pop_mean_15ped
Среднее у открытых (0): 0.0223
Среднее у закрытых (1): 0.0127
P-value: 0.000244
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые объекты были значимо дешевле (даже после выравнивания инфляции).
------------------------------------------------------------
Метрика: ratio_subs_year_adj_to_rent_2026_price_per_m2_mean_15ped
Среднее у открытых (0): 32.4783
Среднее у закрытых (1): 21.3500
P-value: 0.000002
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые объекты были значимо дешевле (даже после выравнивания инфляции).
------------------------------------------------------------
Метрика: ratio_subs_year_adj_to_buy_2026_price_per_m2_mean_15ped
Среднее у открытых (0): 0.1302
Среднее у закрытых (1): 0.0871
P-value: 0.000002
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закр

## Гипотеза 2. Для небольших студий эконом- и комфорт-класса локация в историческом или престижном деловом центре Москвы является фактором повышенного риска из-за дисбаланса между арендной ставкой (Rent Gap) и генерируемой выручкой

Прокси:
- 'dist_to_center_drive_m'
- 'dist_to_center_walk_m'
- 'subs_month'

In [96]:
df2=df.copy()
df2 = df2[['last_seen',
           'content_studio',
           'dist_to_center_drive_m',
            'dist_to_center_walk_m',
            'subs_month',
            'is_closed']]

df2 = df2[df2['content_studio']==1].dropna(subset='subs_month')

df2['is_closed'].value_counts()

is_closed
0    588
1    199
Name: count, dtype: int64

In [97]:
round(df2.groupby(by='is_closed')[['dist_to_center_drive_m',
            'dist_to_center_walk_m',]].mean()/1000, 2)

,dist_to_center_drive_m,dist_to_center_walk_m
is_closed,,
0,12.63,11.95
1,13.37,12.71


In [98]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 787 entries, 0 to 4916
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   last_seen               787 non-null    datetime64[ns]
 1   content_studio          787 non-null    int64         
 2   dist_to_center_drive_m  783 non-null    float64       
 3   dist_to_center_walk_m   787 non-null    float64       
 4   subs_month              787 non-null    float64       
 5   is_closed               787 non-null    int64         
dtypes: datetime64[ns](1), float64(3), int64(2)
memory usage: 43.0 KB


In [99]:
round(df2.groupby(by='last_seen')[['subs_month']].mean()/1000, 2)

,subs_month
last_seen,
2024-02-22,4.40
2024-07-25,6.37
2024-12-13,6.19
2025-02-07,5.49
2025-08-27,10.30
2025-11-17,8.49
2026-02-17,6.16
2026-03-24,10.37


In [100]:
df2.groupby(by='last_seen')[['subs_month']].count()

,subs_month
last_seen,
2024-02-22,22
2024-07-25,31
2024-12-13,15
2025-02-07,46
2025-08-27,25
2025-11-17,34
2026-02-17,26
2026-03-24,588


In [103]:
# Предполагается, что df2 у вас уже создан по вашему коду выше
df2 = df2.copy()
df2['last_seen'] = pd.to_datetime(df2['last_seen'])

# ---------------------------------------------------------
# 1. КОРРЕКТИРОВКА НА ИНФЛЯЦИЮ (Приведение к 2026 году)
# ---------------------------------------------------------
base_date = df2['last_seen'].max()
annual_inflation = 0.14 # Обоснованная ставка 14%

df2['years_diff'] = (base_date - df2['last_seen']).dt.days / 365.25

df2['subs_month_adj'] = df2['subs_month'] * ((1 + annual_inflation) ** df2['years_diff'])

# ---------------------------------------------------------
# 2. ВЫДЕЛЕНИЕ ЭКОНОМ- И КОМФОРТ-КЛАССА
# ---------------------------------------------------------
# Находим границу цены, отсекающую самые дорогие (премиальные) студии.
# Возьмем 75-й перцентиль (то есть оставляем 75% самых доступных студий)
threshold_price = df2['subs_month_adj'].quantile(0.75)

# Фильтруем датасет: оставляем только эконом и комфорт
df_eco_comfort = df2[df2['subs_month_adj'] <= threshold_price].copy()

print(f"=== Сегментация ===")
print(f"Порог отсечения премиум-сегмента (75-й перцентиль): {threshold_price:.2f} руб.")
print(f"Всего студий в выборке: {len(df2)}")
print(f"Студий эконом/комфорт класса для теста: {len(df_eco_comfort)}\n")

# ---------------------------------------------------------
# 3. СТАТИСТИЧЕСКОЕ ТЕСТИРОВАНИЕ (Манн-Уитни)
# ---------------------------------------------------------
dist_metrics = ['dist_to_center_walk_m', 'dist_to_center_drive_m']

print("=== Результаты статистического теста Манна-Уитни ===")
print("H0: Распределения расстояний равны.")
print("H1: Закрытые студии располагались БЛИЖЕ к центру (расстояние меньше).\n")

for col in dist_metrics:
    # Очищаем от пустых значений конкретную метрику
    valid_data = df_eco_comfort.dropna(subset=[col, 'is_closed'])
    
    closed = valid_data[valid_data['is_closed'] == 1][col]
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    
    # alternative='less' -> проверяем, что closed < open_obj (ближе к центру)
    stat, p_value = mannwhitneyu(closed, open_obj, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее расстояние у открытых (0): {open_obj.mean():.1f} метров")
    print(f"Среднее расстояние у закрытых (1): {closed.mean():.1f} метров")
    print(f"P-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: Закрытые студии эконом/комфорт класса действительно располагались значимо БЛИЖЕ к центру.")
    else:
        print("❌ Значимых различий нет.")
    print("-" * 60)

# ---------------------------------------------------------
# 4. КОРРЕЛЯЦИЯ СПИРМЕНА
# ---------------------------------------------------------
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
corr_cols = dist_metrics + ['subs_month_adj']
correlations = df_eco_comfort[corr_cols + ['is_closed']].corr(method='spearman')['is_closed'].sort_values()

print(correlations.drop('is_closed'))

=== Сегментация ===
Порог отсечения премиум-сегмента (75-й перцентиль): 10076.07 руб.
Всего студий в выборке: 787
Студий эконом/комфорт класса для теста: 590

=== Результаты статистического теста Манна-Уитни ===
H0: Распределения расстояний равны.
H1: Закрытые студии располагались БЛИЖЕ к центру (расстояние меньше).

Метрика: dist_to_center_walk_m
Среднее расстояние у открытых (0): 12683.1 метров
Среднее расстояние у закрытых (1): 13572.8 метров
P-value: 0.833539
❌ Значимых различий нет.
------------------------------------------------------------
Метрика: dist_to_center_drive_m
Среднее расстояние у открытых (0): 13348.6 метров
Среднее расстояние у закрытых (1): 14201.6 метров
P-value: 0.807091
❌ Значимых различий нет.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
subs_month_adj           -0.149980
dist_to_center_drive_m    0.035844
dist_to_center_walk_m     0.039885
Name: is_closed, dtype: float64


## Гипотеза 3. Изохрона 5 мин. Расположение в 5-минутной доступности от объектов смежного спроса (кофейни, салоны красоты, магазины здорового питания) или интеграция в ТЦ снижает вероятность банкротства за счет смежной целевой аудтории

In [231]:
df3=df.copy()
df3 = df[['last_seen',

'beauty_count_5ped',
'beauty_reviews_sum_5ped',
'beauty_reviews_mean_5ped',
'bus_count_5ped',
'cafe_count_5ped',
'cafe_reviews_mean_5ped',
'coffee_count_5ped',
'coffee_reviews_sum_5ped',
'coffee_reviews_mean_5ped',
'supermarkets4rich_count_5ped',
'supermarkets4rich_reviews_mean_5ped',
'workplaces_count_5ped',
'workplaces_reviews_sum_5ped',
'workplaces_reviews_mean_5ped',
'metro_count_5ped',
'molls_count_5ped',
'molls_reviews_sum_5ped',
'molls_branches_sum_5ped',
'molls_floors_sum_5ped',
'molls_parking_sum_5ped',
         'is_closed']]

df3['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [232]:
df3['pois_5ped'] = df[['beauty_count_5ped', 'cafe_count_5ped', 'coffee_count_5ped', 'supermarkets4rich_count_5ped']].sum(axis=1)
df3['reviews_5ped'] = df[['beauty_reviews_sum_5ped', 'cafe_reviews_sum_5ped', 'coffee_reviews_sum_5ped', 'supermarkets4rich_reviews_sum_5ped']].sum(axis=1)

# Список метрик для финальной проверки
agglomeration_metrics = [
    'pois_5ped', 
    'reviews_5ped', 
    'molls_count_5ped', 
    'workplaces_count_5ped',
    'workplaces_reviews_sum_5ped',
    'metro_count_5ped',
    'molls_reviews_sum_5ped',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df3.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df3[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: pois_5ped
Среднее Открытые: 16.64 | Среднее Закрытые: 15.44
P-value: 0.009256
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: pois_5ped значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: reviews_5ped
Среднее Открытые: 722.05 | Среднее Закрытые: 616.64
P-value: 0.004342
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: reviews_5ped значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: molls_count_5ped
Среднее Открытые: 0.07 | Среднее Закрытые: 0.05
P-value: 0.000923
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: molls_count_5ped значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: workplaces_count_5ped
Среднее Открытые: 1.16 | Среднее Закрытые: 1.13
P-value: 0.815077
❌ Значимых различий по метрике workplaces_count_5ped не обнаружено.
------------------------------------------------------------
Метрика: workplaces_reviews_su

C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\1967468371.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['pois_5ped'] = df[['beauty_count_5ped', 'cafe_count_5ped', 'coffee_count_5ped', 'supermarkets4rich_count_5ped']].sum(axis=1)
C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\1967468371.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['reviews_5ped'] = df[['beauty_reviews_sum_5ped', 'cafe_reviews_sum_5ped', 'coffee_reviews_sum_5ped', 'supermarkets4rich_reviews_sum_5ped']].sum(axis=1)

## Гипотеза 4. Парки и walkability

In [123]:
df4=df.copy()
df4 = df[['last_seen',
# 'content_class_area',
# 'content_studio',
# 'net',
'parks_count_15ped',
'parks_reviews_sum_15ped',
'parks_count_15drive',
'parks_reviews_sum_15drive',
'green_pct',
'walkability_index_15ped',
'is_closed']]

df4['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [124]:
df4.groupby(by='is_closed')[['green_pct']].mean()

,green_pct
is_closed,
0,9.588047
1,9.665148


In [125]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu



# Список метрик для финальной проверки
agglomeration_metrics = [
'parks_count_15ped',
'parks_reviews_sum_15ped',
'parks_count_15drive',
'parks_reviews_sum_15drive',
'green_pct',
'walkability_index_15ped',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df4.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df4[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: parks_count_15ped
Среднее Открытые: 0.14 | Среднее Закрытые: 0.17
P-value: 0.009604
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: parks_count_15ped значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: parks_reviews_sum_15ped
Среднее Открытые: 30.97 | Среднее Закрытые: 36.54
P-value: 0.008584
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: parks_reviews_sum_15ped значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: parks_count_15drive
Среднее Открытые: 7.01 | Среднее Закрытые: 7.15
P-value: 0.206698
❌ Значимых различий по метрике parks_count_15drive не обнаружено.
------------------------------------------------------------
Метрика: parks_reviews_sum_15drive
Среднее Открытые: 2043.95 | Среднее Закрытые: 2181.85
P-value: 0.090104
❌ Значимых различий по метрике parks_reviews_sum_15drive не обнаружено.
----------------------------------------------------

## Гипотеза 5. Влияние сетевого фактора

In [134]:
df5=df.copy()
df5 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'is_closed']]
df5s = df5[df5['content_studio']==1]
df5cl = df5[df5['content_class_area']==1]
df5['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [131]:
# Список метрик для финальной проверки
agglomeration_metrics = [
'net',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df5.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df5[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: net
Среднее Открытые: 0.81 | Среднее Закрытые: 0.57
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: net значимо выше у выживших объектов.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
net         -0.255269
is_closed    1.000000
Name: is_closed, dtype: float64


### Студии

In [132]:
# Список метрик для финальной проверки
agglomeration_metrics = [
'net',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df5s.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df5s[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: net
Среднее Открытые: 0.92 | Среднее Закрытые: 0.56
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: net значимо выше у выживших объектов.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
net         -0.417563
is_closed    1.000000
Name: is_closed, dtype: float64


### Крупные фтнес-объекты

In [135]:
# Список метрик для финальной проверки
agglomeration_metrics = [
'net',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df5cl.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df5cl[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: net
Среднее Открытые: 0.94 | Среднее Закрытые: 0.50
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: net значимо выше у выживших объектов.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
net         -0.452842
is_closed    1.000000
Name: is_closed, dtype: float64


## Гипотеза 6. Этаж

In [137]:
df6=df.copy()
df6 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'floor_count',
'floor_min',
'floor_max',
'is_closed']]
df6s = df6[df6['content_studio']==1]
df6cl = df6[df6['content_class_area']==1]
df6['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [138]:
df6.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4996 entries, 0 to 4995
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   last_seen           4996 non-null   datetime64[ns]
 1   content_class_area  4996 non-null   int64         
 2   content_studio      4996 non-null   int64         
 3   net                 4996 non-null   int64         
 4   floor_count         1463 non-null   float64       
 5   floor_min           1448 non-null   float64       
 6   floor_max           1448 non-null   float64       
 7   is_closed           4996 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(4)
memory usage: 312.4 KB


In [142]:
df6.groupby(by='is_closed')[['floor_count','floor_min','floor_max']].mean()

,floor_count,floor_min,floor_max
is_closed,,,
0,1.071835,1.664761,1.732309
1,1.175439,1.642857,1.642857


In [146]:
# Список метрик для финальной проверки
agglomeration_metrics = [
'floor_count',
'floor_min',
'floor_max',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df6.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df6[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: floor_count
Среднее Открытые: 1.07 | Среднее Закрытые: 1.18
P-value: 0.788941
❌ Значимых различий по метрике floor_count не обнаружено.
------------------------------------------------------------
Метрика: floor_min
Среднее Открытые: 1.66 | Среднее Закрытые: 1.64
P-value: 0.709469
❌ Значимых различий по метрике floor_min не обнаружено.
------------------------------------------------------------
Метрика: floor_max
Среднее Открытые: 1.73 | Среднее Закрытые: 1.64
P-value: 0.599527
❌ Значимых различий по метрике floor_max не обнаружено.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
floor_max      0.006623
floor_min      0.014502
floor_count    0.020983
is_closed      1.000000
Name: is_closed, dtype: float64


## Гипотеза 7. Цифровой след

In [189]:
df7=df.copy()
df7 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'is_advertised',
'photos',
'hours_per_day',
'parking',
'free_parking',
'payed_parking',
'nearest_station_d',
'accessible_entrance',
'online_training',
'is_closed']]
df7 = df7.dropna(subset='is_advertised')
df7s = df7[df7['content_studio']==1].dropna(subset='is_advertised')
df7cl = df7[df7['content_class_area']==1].dropna(subset='is_advertised')
df7['is_closed'].value_counts()

is_closed
0    1964
1      82
Name: count, dtype: int64

In [190]:
df7[df7['last_seen']>'01-01-2026'].info()

<class 'pandas.core.frame.DataFrame'>
Index: 2046 entries, 2 to 4990
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   last_seen            2046 non-null   datetime64[ns]
 1   content_class_area   2046 non-null   int64         
 2   content_studio       2046 non-null   int64         
 3   net                  2046 non-null   int64         
 4   is_advertised        2046 non-null   float64       
 5   photos               1900 non-null   float64       
 6   hours_per_day        2018 non-null   float64       
 7   parking              2046 non-null   float64       
 8   free_parking         2046 non-null   float64       
 9   payed_parking        2046 non-null   float64       
 10  nearest_station_d    2038 non-null   float64       
 11  accessible_entrance  2046 non-null   int64         
 12  online_training      2046 non-null   int64         
 13  is_closed            2046 non-null   i

In [192]:
# Список метрик для финальной проверки
agglomeration_metrics = [
'is_advertised',
'photos',
'hours_per_day',
'parking',
'free_parking',
'payed_parking',
'nearest_station_d',
'accessible_entrance',
'online_training',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df7.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df7[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: is_advertised
Среднее Открытые: 0.10 | Среднее Закрытые: 0.05
P-value: 0.054224
❌ Значимых различий по метрике is_advertised не обнаружено.
------------------------------------------------------------
Метрика: photos
Среднее Открытые: 1.00 | Среднее Закрытые: 1.00
P-value: 1.000000
❌ Значимых различий по метрике photos не обнаружено.
------------------------------------------------------------
Метрика: hours_per_day
Среднее Открытые: 13.74 | Среднее Закрытые: 11.94
P-value: 0.000118
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: hours_per_day значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: parking
Среднее Открытые: 125.26 | Среднее Закрытые: 82.09
P-value: 0.731593
❌ Значимых различий по метрике parking не обнаружено.
------------------------------------------------------------
Метрика: free_parking
Среднее Открытые: 60.32 | Среднее Закрытые: 68.52
P-value: 0.902356
❌ Значимы

In [227]:
df7=df.copy()
df7 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'rating',
'rating_cnt',
'phones_true',
'emails_true',
'urls_true',
'content_trainings',
'is_closed']]
df7 = df7
df7s = df7[df7['content_studio']==1]
df7cl = df7[df7['content_class_area']==1]
df7['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [228]:
# Список метрик для финальной проверки
agglomeration_metrics = [

'phones_true',
'emails_true',
'urls_true',
'content_trainings',
'rating',
'rating_cnt',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df7.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df7[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: phones_true
Среднее Открытые: 0.97 | Среднее Закрытые: 0.93
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: phones_true значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: emails_true
Среднее Открытые: 0.09 | Среднее Закрытые: 0.04
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: emails_true значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: urls_true
Среднее Открытые: 0.84 | Среднее Закрытые: 0.72
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: urls_true значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: content_trainings
Среднее Открытые: 5.73 | Среднее Закрытые: 3.54
P-value: 0.000000
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: content_trainings значимо выше у выживших объектов.
------------------------------------------------------------
Метрика: rating
Среднее Открытые: 4.53 | 

## Гипотеза 8. Конкуренты. Когда нет конкурентов - это плохо, когда их очень много - тоже плохо.
- 'competitors_count_15ped' - все конкуренты (вкл. объекты своей сети), которые были либо часть времени жизни объекта или всегда
- 'competitors_during_all_comp_count_15ped', - конкуренты, которые были всегда
- 'competitors_during_part_comp_count_15ped', - конкуренты, которые были часть времени жизни объекта
- 'competitors_during_all_cann_count_15ped', - объекты своей сети, которые были всегда
- 'competitors_during_part_cann_count_15ped', - объекты своей сети, которые были часть времени жизни объекта

- 15_ped - пешеходная изохрона 15 мин
- 15_drive - автомобильная изохрона 15 мин

In [204]:
df8=df.copy()
df8 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_during_all_cann_count_15ped',
'competitors_during_part_cann_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',
'competitors_during_all_cann_count_15drive',
'competitors_during_part_cann_count_15drive',
'competitors_comp_cnt_15drive',
'competitors_comp_cnt_15ped',
'is_closed']]
df8 = df8
df8s = df8[df8['content_studio']==1]
df8cl = df8[df8['content_class_area']==1]
df8['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [205]:
df8.groupby(by='is_closed')[['competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_during_all_cann_count_15ped',
'competitors_during_part_cann_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',
'competitors_during_all_cann_count_15drive',
'competitors_during_part_cann_count_15drive',
'competitors_comp_cnt_15drive',
'competitors_comp_cnt_15ped',]].mean()

,competitors_count_15ped,competitors_during_all_comp_count_15ped,competitors_during_part_comp_count_15ped,competitors_during_all_cann_count_15ped,competitors_during_part_cann_count_15ped,competitors_count_15drive,competitors_during_all_comp_count_15drive,competitors_during_part_comp_count_15drive,competitors_during_all_cann_count_15drive,competitors_during_part_cann_count_15drive,competitors_comp_cnt_15drive,competitors_comp_cnt_15ped
is_closed,,,,,,,,,,,,
0,17.380132,7.948608,4.941717,0.058283,0.022107,129.585128,59.129486,36.400804,0.224519,0.062590,0.457834,0.455253
1,17.965631,8.754131,5.044944,0.046927,0.013880,132.046266,64.455387,37.337740,0.142763,0.033047,0.483008,0.483724


In [206]:
# Список метрик для финальной проверки
agglomeration_metrics = [

'competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_during_all_cann_count_15ped',
'competitors_during_part_cann_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',
'competitors_during_all_cann_count_15drive',
'competitors_during_part_cann_count_15drive',
'competitors_comp_cnt_15drive',
'competitors_comp_cnt_15ped',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df8.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо ниже у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
Метрика: competitors_comp_cnt_15drive
Среднее Открытые: 0.46 | Среднее Закрытые: 0.48
P-value: 0.000001
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: competitors_comp_cnt_15drive значимо ниже у выживших объектов.
------------------------------------------------------------
Метрика: competitors_comp_cnt_15ped
Среднее Открытые: 0.46 | Среднее Закрытые: 0.48
P-value: 0.000032
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: competitors_comp_cnt_15ped значимо ниже у выживших объектов.
------------------------------------------------------------print(df8[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: competitors_count_15ped
Среднее Открытые: 17.38 | Среднее Закрытые: 17.97
P-value: 0.082487
❌ Значимых различий по метрике competitors_count_15ped не обнаружено.
------------------------------------------------------------
Метрика: competitors_during_all_comp_count_15ped
Среднее Открытые: 7.95 | Среднее Закрытые: 8.75
P-value: 0.000041
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: competitors_during_all_comp_count_15ped значимо ниже у выживших объектов.
------------------------------------------------------------
Метрика: competitors_during_part_comp_count_15ped
Среднее Открытые: 4.94 | Среднее Закрытые: 5.04
P-value: 0.110302
❌ Значимых различий по метрике competitors_during_part_comp_count_15ped не обнаружено.
------------------------------------------------------------
Метрика: competitors_during_all_cann_count_15ped
Среднее Открытые: 0.06 | Среднее Закрытые: 0.05
P-value: 0.896006
❌ Значимых различий по метрике competitors_du

### Полное отсутствие конкурентов (0) - это плохо для выживаемости, и очень много конкурентов (4+) - тоже плохо для выживаемости

In [174]:
df8=df.copy()
df8 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_during_all_cann_count_15ped',
'competitors_during_part_cann_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',
'competitors_during_all_cann_count_15drive',
'competitors_during_part_cann_count_15drive',
'is_closed']]

df8['is_closed'].value_counts()

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [184]:
# Ваш df8 уже создан, копируем его чтобы не менять оригинал
df8_test = df8.copy()

# Список колонок с конкурентами
comp_cols = [
    'competitors_count_15ped', 'competitors_during_all_comp_count_15ped',
    'competitors_during_part_comp_count_15ped', 'competitors_during_all_cann_count_15ped',
    'competitors_during_part_cann_count_15ped',
    'competitors_count_15drive', 'competitors_during_all_comp_count_15drive',
    'competitors_during_part_comp_count_15drive', 'competitors_during_all_cann_count_15drive',
    'competitors_during_part_cann_count_15drive',
]

print("=== Проверка нелинейной гипотезы конкуренции (U-образная кривая смертности) ===\n")

for col in comp_cols:
    if col not in df8_test.columns:
        continue
        
    valid_data = df8_test.dropna(subset=[col, 'is_closed']).copy()
    
    # Разбиваем на 3 группы
    conditions = [
        valid_data[col] == 0,
        (valid_data[col] >= 1) & (valid_data[col] <= 3),
        valid_data[col] >= 4
    ]
    choices = ['0', '1-3', '4+']
    valid_data['comp_group'] = np.select(conditions, choices, default='Unknown')
    
    # Считаем статистику по группам
    stats = valid_data.groupby('comp_group')['is_closed'].agg(['count', 'sum'])
    stats['closure_rate'] = stats['sum'] / stats['count'] # Доля закрывшихся
    
    # Проверяем, есть ли все три группы в данных (иногда 4+ может не быть)
    if '0' in stats.index and '1-3' in stats.index and '4+' in stats.index:
        
        rate_0 = stats.loc['0', 'closure_rate']
        rate_1_3 = stats.loc['1-3', 'closure_rate']
        rate_4 = stats.loc['4+', 'closure_rate']
        
        # Z-тест: Доля закрытий при 0 конкурентах > чем при 1-3?
        count_0_vs_13 = np.array([stats.loc['0', 'sum'], stats.loc['1-3', 'sum']])
        nobs_0_vs_13 = np.array([stats.loc['0', 'count'], stats.loc['1-3', 'count']])
        _, p_val_0 = proportions_ztest(count_0_vs_13, nobs_0_vs_13, alternative='larger')
        
        # Z-тест: Доля закрытий при 4+ конкурентах > чем при 1-3?
        count_4_vs_13 = np.array([stats.loc['4+', 'sum'], stats.loc['1-3', 'sum']])
        nobs_4_vs_13 = np.array([stats.loc['4+', 'count'], stats.loc['1-3', 'count']])
        _, p_val_4 = proportions_ztest(count_4_vs_13, nobs_4_vs_13, alternative='larger')
        
        print(f"Метрика: {col}")
        print(f"Доля закрытий [0 конкур.]:  {rate_0:.1%} (объектов: {stats.loc['0', 'count']})")
        print(f"Доля закрытий [1-3 конкур.]: {rate_1_3:.1%} (объектов: {stats.loc['1-3', 'count']})")
        print(f"Доля закрытий [4+ конкур.]: {rate_4:.1%} (объектов: {stats.loc['4+', 'count']})")
        
        # Делаем выводы
        is_u_shape = (rate_0 > rate_1_3) and (rate_4 > rate_1_3)
        is_sig = (p_val_0 < 0.05) and (p_val_4 < 0.05)
        
        if is_u_shape and is_sig:
            print("✅ ИДЕАЛЬНО ПОДТВЕРЖДАЕТСЯ: Изоляция (0) и перенасыщение (4+) значимо повышают риск закрытия.")
        elif is_u_shape:
            print("⚠️ ТЕНДЕНЦИЯ ЕСТЬ: U-образная кривая наблюдается (1-3 самое безопасное), но не хватает стат. значимости.")
            print(f"   p-value (0 vs 1-4): {p_val_0:.4f} | p-value (4+ vs 1-3): {p_val_4:.4f}")
        else:
            print("❌ Гипотеза не подтвердилась (нет U-образной формы).")
            
    else:
        print(f"Метрика: {col}")
        print("❌ Недостаточно данных для формирования всех трех групп (0, 1-3, 4+).")
        
    print("-" * 70)

=== Проверка нелинейной гипотезы конкуренции (U-образная кривая смертности) ===

Метрика: competitors_count_15ped
Доля закрытий [0 конкур.]:  23.2% (объектов: 99)
Доля закрытий [1-3 конкур.]: 26.6% (объектов: 218)
Доля закрытий [4+ конкур.]: 30.6% (объектов: 4679)
❌ Гипотеза не подтвердилась (нет U-образной формы).
----------------------------------------------------------------------
Метрика: competitors_during_all_comp_count_15ped
Доля закрытий [0 конкур.]:  26.6% (объектов: 241)
Доля закрытий [1-3 конкур.]: 27.3% (объектов: 1099)
Доля закрытий [4+ конкур.]: 31.4% (объектов: 3656)
❌ Гипотеза не подтвердилась (нет U-образной формы).
----------------------------------------------------------------------
Метрика: competitors_during_part_comp_count_15ped
Доля закрытий [0 конкур.]:  28.2% (объектов: 815)
Доля закрытий [1-3 конкур.]: 30.6% (объектов: 1774)
Доля закрытий [4+ конкур.]: 30.7% (объектов: 2407)
❌ Гипотеза не подтвердилась (нет U-образной формы).
--------------------------------

## Гипотеза 9. Население и разные форматы
- 'INHAB_count_15ped'
- 'INHAB_count_15drive'

- 15_ped - пешеходная изохрона 15 мин
- 15_drive - автомобильная изохрона 15 мин

In [224]:
df9=df.copy()
df9 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'INHAB_count_15ped',
'INHAB_count_15drive',
'workplaces_count_15ped',
'workplaces_count_15drive',
'workplaces_reviews_sum_15ped',
'competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',

'is_closed']]
df9['competitors_count_15ped']=df9['INHAB_count_15ped']/(df9['competitors_count_15ped']+1)
df9['competitors_during_all_comp_count_15ped']=df9['INHAB_count_15ped']/(df9['competitors_during_all_comp_count_15ped']+1)
df9['competitors_during_part_comp_count_15ped']=df9['INHAB_count_15ped']/(df9['competitors_during_part_comp_count_15ped']+1)

df9['competitors_count_15drive']=df9['INHAB_count_15drive']/(df9['competitors_count_15drive']+1)
df9['competitors_during_all_comp_count_15drive']=df9['INHAB_count_15drive']/(df9['competitors_during_all_comp_count_15drive']+1)
df9['competitors_during_part_comp_count_15drive']=df9['INHAB_count_15drive']/(df9['competitors_during_part_comp_count_15drive']+1)


df9s = df9[df9['content_studio']==1]
df9cl = df9[df9['content_class_area']==1]

df9['is_closed'].value_counts()

C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\2879451366.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df9['competitors_count_15ped']=df9['INHAB_count_15ped']/(df9['competitors_count_15ped']+1)
C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\2879451366.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df9['competitors_during_all_comp_count_15ped']=df9['INHAB_count_15ped']/(df9['competitors_during_all_comp_count_15ped']+1)
C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\2879451366.py:21:

is_closed
0    3483
1    1513
Name: count, dtype: int64

In [226]:
# Список метрик для финальной проверки
agglomeration_metrics = [

'INHAB_count_15ped',
'INHAB_count_15drive',
'workplaces_count_15ped',
'workplaces_count_15drive',
'workplaces_reviews_sum_15ped',
'competitors_count_15ped',
'competitors_during_all_comp_count_15ped',
'competitors_during_part_comp_count_15ped',
'competitors_count_15drive',
'competitors_during_all_comp_count_15drive',
'competitors_during_part_comp_count_15drive',

]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df9.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df9[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: INHAB_count_15ped
Среднее Открытые: 25651.47 | Среднее Закрытые: 26076.58
P-value: 0.843349
❌ Значимых различий по метрике INHAB_count_15ped не обнаружено.
------------------------------------------------------------
Метрика: INHAB_count_15drive
Среднее Открытые: 703901.17 | Среднее Закрытые: 718614.36
P-value: 0.980054
❌ Значимых различий по метрике INHAB_count_15drive не обнаружено.
------------------------------------------------------------
Метрика: workplaces_count_15ped
Среднее Открытые: 8.90 | Среднее Закрытые: 9.25
P-value: 0.883769
❌ Значимых различий по метрике workplaces_count_15ped не обнаружено.
------------------------------------------------------------
Метрика: workplaces_count_15drive
Среднее Открытые: 277.14 | Среднее Закрытые: 284.12
P-value: 0.819013
❌ Значимых различий по метрике workplaces_count_15drive не обнаружено.
------------------------------------------------------------
Метрика: wo

## Гипотеза 10. Индексы

In [235]:
df.shape

(4996, 333)

In [236]:
df10=df.copy()
df10 = df[['last_seen',
'content_class_area',
'content_studio',
'net',
'entropy_15ped',
'clustering_index_15ped',
'walkability_index_15ped',
'bus_count_15ped',
'metro_count_15ped',
'is_closed']]
df10['betweenness'] = df10['metro_count_15ped']+df10['bus_count_15ped']
df10s = df10[df10['content_studio']==1]
df10cl = df10[df10['content_class_area']==1]
df10['is_closed'].value_counts()

C:\Users\Mariia\AppData\Local\Temp\ipykernel_27548\451397022.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df10['betweenness'] = df10['metro_count_15ped']+df10['bus_count_15ped']


is_closed
0    3483
1    1513
Name: count, dtype: int64

In [237]:
# Список метрик для финальной проверки
agglomeration_metrics = [

'clustering_index_15ped',


]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df10.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='greater')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо выше у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df10[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: clustering_index_15ped
Среднее Открытые: 15.87 | Среднее Закрытые: 14.21
P-value: 0.000035
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: clustering_index_15ped значимо выше у выживших объектов.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
clustering_index_15ped   -0.056216
is_closed                 1.000000
Name: is_closed, dtype: float64


In [239]:
# Список метрик для финальной проверки
agglomeration_metrics = [

'entropy_15ped',
'walkability_index_15ped',
'betweenness'
]

print("=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===\n")

for col in agglomeration_metrics:
    valid_data = df10.dropna(subset=[col, 'is_closed'])
    open_obj = valid_data[valid_data['is_closed'] == 0][col]
    closed = valid_data[valid_data['is_closed'] == 1][col]
    
    # Проверяем, что у открытых значения БОЛЬШЕ (alternative='greater')
    stat, p_value = mannwhitneyu(open_obj, closed, alternative='less')
    
    print(f"Метрика: {col}")
    print(f"Среднее Открытые: {open_obj.mean():.2f} | Среднее Закрытые: {closed.mean():.2f}")
    print(f"P-value: {p_value:.6f}")
    if p_value < 0.05:
        print(f"✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: {col} значимо ниже у выживших объектов.")
    else:
        print(f"❌ Значимых различий по метрике {col} не обнаружено.")
    print("-" * 60)

# Корреляция Спирмена
print("\n=== Корреляция Спирмена с таргетом (is_closed) ===")
print(df10[agglomeration_metrics + ['is_closed']].corr(method='spearman')['is_closed'].sort_values())

=== Проверка Гипотезы 5 (Интегральные показатели агломерации) ===

Метрика: entropy_15ped
Среднее Открытые: 0.74 | Среднее Закрытые: 0.75
P-value: 0.013397
✅ Гипотеза ПОДТВЕРЖДАЕТСЯ: entropy_15ped значимо ниже у выживших объектов.
------------------------------------------------------------
Метрика: walkability_index_15ped
Среднее Открытые: -0.03 | Среднее Закрытые: 0.07
P-value: 0.164735
❌ Значимых различий по метрике walkability_index_15ped не обнаружено.
------------------------------------------------------------
Метрика: betweenness
Среднее Открытые: 18.09 | Среднее Закрытые: 18.39
P-value: 0.216349
❌ Значимых различий по метрике betweenness не обнаружено.
------------------------------------------------------------

=== Корреляция Спирмена с таргетом (is_closed) ===
betweenness                0.011101
walkability_index_15ped    0.013798
entropy_15ped              0.031334
is_closed                  1.000000
Name: is_closed, dtype: float64
